In [1]:
from pathlib import Path
import duckdb
import pandas as pd

# =========================================================
# PATHS
# =========================================================

# =========================================================
# PROJECT ROOT
# =========================================================

import os
from pathlib import Path


def resolve_project_root():

    # Used automatically by run_pipeline.py
    env_root = os.environ.get(
        "DATA_ENGINEERING_R62_ROOT"
    )

    if env_root:
        root = Path(env_root).resolve()

        if (
            (root / "data").exists()
            and
            (root / "notebooks").exists()
        ):
            return root


    # Interactive notebook execution
    cwd = Path.cwd().resolve()

    candidates = [
        cwd,
        cwd.parent
    ]

    for candidate in candidates:

        if (
            (candidate / "data").exists()
            and
            (candidate / "notebooks").exists()
        ):
            return candidate


    raise RuntimeError(
        "Could not locate DATA_ENGINEERING_R62 project root."
    )


PROJECT_ROOT = resolve_project_root()

print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "\n✅ SUCCESS — Project root resolved."
)
PROCESSED = PROJECT_ROOT / "data" / "processed"

EMAIL_DAILY = PROCESSED / "email_daily_r62.parquet"
USB_DAILY = PROCESSED / "usb_daily_r62.parquet"
HTTP_DAILY = PROCESSED / "http_daily_r62.parquet"
LOGON_DAILY = PROCESSED / "logon_daily_r62.parquet"

USER_DAY = PROCESSED / "user_day_r62_v1.parquet"

email_daily_path = str(EMAIL_DAILY).replace("\\", "/")
usb_daily_path = str(USB_DAILY).replace("\\", "/")
http_daily_path = str(HTTP_DAILY).replace("\\", "/")
logon_daily_path = str(LOGON_DAILY).replace("\\", "/")
user_day_path = str(USER_DAY).replace("\\", "/")

for f in [
    EMAIL_DAILY,
    USB_DAILY,
    HTTP_DAILY,
    LOGON_DAILY
]:
    assert f.exists(), f"Missing file: {f}"

print("Email Daily :", EMAIL_DAILY.exists())
print("USB Daily   :", USB_DAILY.exists())
print("HTTP Daily  :", HTTP_DAILY.exists())
print("Logon Daily :", LOGON_DAILY.exists())

print("\nOutput:")
print(USER_DAY)

print("\n✅ SUCCESS — User-Day setup is ready.")

Project root: C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62

✅ SUCCESS — Project root resolved.
Email Daily : True
USB Daily   : True
HTTP Daily  : True
Logon Daily : True

Output:
C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\processed\user_day_r62_v1.parquet

✅ SUCCESS — User-Day setup is ready.


In [2]:
# =========================================================
# BUILD UNIFIED USER-DAY TABLE
# =========================================================

con = duckdb.connect()

con.execute(f"""
COPY (

    WITH all_user_days AS (

        SELECT user, day
        FROM read_parquet('{email_daily_path}')

        UNION

        SELECT user, day
        FROM read_parquet('{usb_daily_path}')

        UNION

        SELECT user, day
        FROM read_parquet('{http_daily_path}')

        UNION

        SELECT user, day
        FROM read_parquet('{logon_daily_path}')

    )

    SELECT

        k.user,
        k.day,

        -- =================================================
        -- CALENDAR
        -- =================================================

        CAST(
            STRFTIME(k.day, '%u')
            AS INTEGER
        ) AS day_of_week,

        CASE
            WHEN CAST(
                STRFTIME(k.day, '%u')
                AS INTEGER
            ) IN (6, 7)
            THEN 1
            ELSE 0
        END AS is_weekend,


        -- =================================================
        -- EMAIL
        -- =================================================

        COALESCE(
            e.email_event_count,
            0
        ) AS email_event_count,

        COALESCE(
            e.email_send_count,
            0
        ) AS email_send_count,

        COALESCE(
            e.email_view_count,
            0
        ) AS email_view_count,

        COALESCE(
            e.external_email_count,
            0
        ) AS external_email_count,

        COALESCE(
            e.internal_email_count,
            0
        ) AS internal_email_count,

        COALESCE(
            e.total_sent_email_bytes,
            0
        ) AS total_sent_email_bytes,

        COALESCE(
            e.external_email_bytes,
            0
        ) AS external_email_bytes,

        COALESCE(
            e.emails_with_attachments,
            0
        ) AS emails_with_attachments,

        COALESCE(
            e.attachment_count,
            0
        ) AS attachment_count,

        COALESCE(
            e.unique_external_recipients,
            0
        ) AS unique_external_recipients,

        COALESCE(
            e.unique_external_domains,
            0
        ) AS unique_external_domains,


        -- =================================================
        -- FILE / USB
        -- =================================================

        COALESCE(
            u.file_event_count,
            0
        ) AS file_event_count,

        COALESCE(
            u.file_copy_count,
            0
        ) AS file_copy_count,

        COALESCE(
            u.file_open_count,
            0
        ) AS file_open_count,

        COALESCE(
            u.file_write_count,
            0
        ) AS file_write_count,

        COALESCE(
            u.file_delete_count,
            0
        ) AS file_delete_count,

        COALESCE(
            u.usb_copy_count,
            0
        ) AS usb_copy_count,

        COALESCE(
            u.from_removable_count,
            0
        ) AS from_removable_count,

        COALESCE(
            u.unique_usb_files,
            0
        ) AS unique_usb_files,

        COALESCE(
            u.unique_usb_extensions,
            0
        ) AS unique_usb_extensions,


        -- =================================================
        -- HTTP
        -- =================================================

        COALESCE(
            h.http_event_count,
            0
        ) AS http_event_count,

        COALESCE(
            h.www_visit_count,
            0
        ) AS www_visit_count,

        COALESCE(
            h.www_download_count,
            0
        ) AS www_download_count,

        COALESCE(
            h.www_upload_count,
            0
        ) AS www_upload_count,

        COALESCE(
            h.unique_domains,
            0
        ) AS unique_domains,

        COALESCE(
            h.unique_upload_domains,
            0
        ) AS unique_upload_domains,


        -- =================================================
        -- LOGON
        -- =================================================

        COALESCE(
            l.logon_count,
            0
        ) AS logon_count,

        COALESCE(
            l.logoff_count,
            0
        ) AS logoff_count,

        l.first_logon,
        l.last_logoff


    FROM all_user_days k

    LEFT JOIN read_parquet(
        '{email_daily_path}'
    ) e
        ON k.user = e.user
       AND k.day = e.day

    LEFT JOIN read_parquet(
        '{usb_daily_path}'
    ) u
        ON k.user = u.user
       AND k.day = u.day

    LEFT JOIN read_parquet(
        '{http_daily_path}'
    ) h
        ON k.user = h.user
       AND k.day = h.day

    LEFT JOIN read_parquet(
        '{logon_daily_path}'
    ) l
        ON k.user = l.user
       AND k.day = l.day

    ORDER BY
        k.user,
        k.day

)

TO '{user_day_path}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD
);
""")

con.close()

assert USER_DAY.exists()

print("Created:")
print(USER_DAY)

print("\n✅ SUCCESS — user_day_r62_v1.parquet created.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Created:
C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\processed\user_day_r62_v1.parquet

✅ SUCCESS — user_day_r62_v1.parquet created.


In [5]:
# =========================================================
# INSPECT USER-DAY
# =========================================================

con = duckdb.connect()

schema = con.execute(f"""
DESCRIBE
SELECT *
FROM read_parquet(
    '{user_day_path}'
)
""").df()

sample = con.execute(f"""
SELECT *
FROM read_parquet(
    '{user_day_path}'
)
ORDER BY user, day
LIMIT 10
""").df()

con.close()

print("=" * 100)
print("USER-DAY SCHEMA")
print("=" * 100)

print(
    schema[
        ["column_name", "column_type"]
    ].to_string(index=False)
)

print("\nTotal columns:", len(schema))

print("\n" + "=" * 100)
print("SAMPLE")
print("=" * 100)

print(
    sample.to_string(index=False)
)

print("\n✅ SUCCESS — User-Day inspected.")

USER-DAY SCHEMA
               column_name column_type
                      user     VARCHAR
                       day        DATE
               day_of_week     INTEGER
                is_weekend     INTEGER
         email_event_count      BIGINT
          email_send_count      DOUBLE
          email_view_count      DOUBLE
      external_email_count      DOUBLE
      internal_email_count      DOUBLE
    total_sent_email_bytes      DOUBLE
      external_email_bytes      DOUBLE
   emails_with_attachments      DOUBLE
          attachment_count      DOUBLE
unique_external_recipients      BIGINT
   unique_external_domains      BIGINT
          file_event_count      BIGINT
           file_copy_count      DOUBLE
           file_open_count      DOUBLE
          file_write_count      DOUBLE
         file_delete_count      DOUBLE
            usb_copy_count      DOUBLE
      from_removable_count      DOUBLE
          unique_usb_files      BIGINT
     unique_usb_extensions      BIGINT
         

In [7]:
# =========================================================
# USER-DAY CORE VALIDATION
# =========================================================

con = duckdb.connect()

summary = con.execute(f"""
SELECT

    COUNT(*) AS rows,

    COUNT(
        DISTINCT user
    ) AS users,

    MIN(day) AS first_day,

    MAX(day) AS last_day

FROM read_parquet(
    '{user_day_path}'
)
""").df()


duplicates = con.execute(f"""
SELECT COUNT(*)

FROM (

    SELECT
        user,
        day,
        COUNT(*) AS n

    FROM read_parquet(
        '{user_day_path}'
    )

    GROUP BY
        user,
        day

    HAVING COUNT(*) > 1
)
""").fetchone()[0]


null_keys = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{user_day_path}'
)

WHERE user IS NULL
   OR day IS NULL
""").fetchone()[0]


negative_values = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{user_day_path}'
)

WHERE
       email_event_count < 0
    OR email_send_count < 0
    OR external_email_count < 0
    OR total_sent_email_bytes < 0
    OR external_email_bytes < 0

    OR file_event_count < 0
    OR file_copy_count < 0
    OR usb_copy_count < 0

    OR http_event_count < 0
    OR www_visit_count < 0
    OR www_download_count < 0
    OR www_upload_count < 0

    OR logon_count < 0
    OR logoff_count < 0
""").fetchone()[0]


con.close()


print("=" * 100)
print("USER-DAY CORE VALIDATION")
print("=" * 100)

print(
    summary.to_string(
        index=False
    )
)

print("\nDuplicate user-days :", duplicates)
print("Null user/day       :", null_keys)
print("Negative values     :", negative_values)


assert duplicates == 0
assert null_keys == 0
assert negative_values == 0
assert int(summary.loc[0, "users"]) == 4000


print(
    "\n✅ SUCCESS — USER-DAY CORE VALIDATION PASSED."
)

USER-DAY CORE VALIDATION
   rows  users  first_day   last_day
1394010   4000 2010-01-02 2011-06-01

Duplicate user-days : 0
Null user/day       : 0
Negative values     : 0

✅ SUCCESS — USER-DAY CORE VALIDATION PASSED.


In [9]:
# =========================================================
# CROSS-CHANNEL TOTAL PRESERVATION
# =========================================================

con = duckdb.connect()

totals = con.execute(f"""
SELECT

    -- EMAIL
    SUM(email_event_count)
        AS email_events,

    SUM(email_send_count)
        AS email_send,

    SUM(email_view_count)
        AS email_view,

    SUM(external_email_count)
        AS external_email,


    -- FILE / USB
    SUM(file_event_count)
        AS file_events,

    SUM(file_copy_count)
        AS file_copy,

    SUM(file_open_count)
        AS file_open,

    SUM(file_write_count)
        AS file_write,

    SUM(file_delete_count)
        AS file_delete,

    SUM(usb_copy_count)
        AS usb_copy,


    -- HTTP
    SUM(http_event_count)
        AS http_events,

    SUM(www_visit_count)
        AS www_visit,

    SUM(www_download_count)
        AS www_download,

    SUM(www_upload_count)
        AS www_upload,


    -- LOGON
    SUM(logon_count)
        AS logon,

    SUM(logoff_count)
        AS logoff

FROM read_parquet(
    '{user_day_path}'
)
""").df()

con.close()


print("=" * 100)
print("CROSS-CHANNEL TOTAL PRESERVATION")
print("=" * 100)

print(
    totals.T.to_string(
        header=False
    )
)


checks = {

    "Email Events":
        int(totals.loc[0, "email_events"])
        == 10_994_957,

    "Email Send":
        int(totals.loc[0, "email_send"])
        == 3_832_466,

    "Email View":
        int(totals.loc[0, "email_view"])
        == 7_162_491,

    "External Email":
        int(totals.loc[0, "external_email"])
        == 1_656_078,


    "File Events":
        int(totals.loc[0, "file_events"])
        == 2_014_883,

    "File Copy":
        int(totals.loc[0, "file_copy"])
        == 725_458,

    "File Open":
        int(totals.loc[0, "file_open"])
        == 657_500,

    "File Write":
        int(totals.loc[0, "file_write"])
        == 251_126,

    "File Delete":
        int(totals.loc[0, "file_delete"])
        == 380_799,

    "USB Copy":
        int(totals.loc[0, "usb_copy"])
        == 333_611,


    "HTTP Events":
        int(totals.loc[0, "http_events"])
        == 117_025_216,

    "WWW Visit":
        int(totals.loc[0, "www_visit"])
        == 115_401_178,

    "WWW Download":
        int(totals.loc[0, "www_download"])
        == 1_413_637,

    "WWW Upload":
        int(totals.loc[0, "www_upload"])
        == 210_401,


    "Logon":
        int(totals.loc[0, "logon"])
        == 1_948_933,

    "Logoff":
        int(totals.loc[0, "logoff"])
        == 1_581_352
}


print("\nChecks:")

for name, passed in checks.items():

    print(
        f"{name:<20}: {passed}"
    )


assert all(checks.values())


print(
    "\n✅ SUCCESS — ALL CHANNEL TOTALS PRESERVED AFTER MERGE."
)

CROSS-CHANNEL TOTAL PRESERVATION
email_events     10994957.0
email_send        3832466.0
email_view        7162491.0
external_email    1656078.0
file_events       2014883.0
file_copy          725458.0
file_open          657500.0
file_write         251126.0
file_delete        380799.0
usb_copy           333611.0
http_events     117025216.0
www_visit       115401178.0
www_download      1413637.0
www_upload         210401.0
logon             1948933.0
logoff            1581352.0

Checks:
Email Events        : True
Email Send          : True
Email View          : True
External Email      : True
File Events         : True
File Copy           : True
File Open           : True
File Write          : True
File Delete         : True
USB Copy            : True
HTTP Events         : True
WWW Visit           : True
WWW Download        : True
WWW Upload          : True
Logon               : True
Logoff              : True

✅ SUCCESS — ALL CHANNEL TOTALS PRESERVED AFTER MERGE.


In [11]:
# =========================================================
# LOGICAL SANITY CHECKS
# =========================================================

con = duckdb.connect()

sanity = con.execute(f"""
SELECT

    SUM(
        CASE
            WHEN external_email_count
                 > email_send_count
            THEN 1
            ELSE 0
        END
    ) AS bad_external_email,

    SUM(
        CASE
            WHEN internal_email_count
                 + external_email_count
                 <> email_send_count
            THEN 1
            ELSE 0
        END
    ) AS bad_email_partition,

    SUM(
        CASE
            WHEN usb_copy_count
                 > file_copy_count
            THEN 1
            ELSE 0
        END
    ) AS bad_usb_copy,

    SUM(
        CASE
            WHEN www_visit_count
                 + www_download_count
                 + www_upload_count
                 <> http_event_count
            THEN 1
            ELSE 0
        END
    ) AS bad_http_partition,

    SUM(
        CASE
            WHEN unique_upload_domains
                 > unique_domains
            THEN 1
            ELSE 0
        END
    ) AS bad_upload_domains

FROM read_parquet(
    '{user_day_path}'
)
""").df()

con.close()


print("=" * 100)
print("LOGICAL SANITY CHECKS")
print("=" * 100)

print(
    sanity.to_string(
        index=False
    )
)


assert (
    sanity.iloc[0] == 0
).all()


print(
    "\n✅ SUCCESS — ALL USER-DAY LOGICAL CHECKS PASSED."
)

LOGICAL SANITY CHECKS
 bad_external_email  bad_email_partition  bad_usb_copy  bad_http_partition  bad_upload_domains
                0.0                  0.0           0.0                 0.0                 0.0

✅ SUCCESS — ALL USER-DAY LOGICAL CHECKS PASSED.


In [13]:
# =========================================================
# FINAL USER-DAY SUMMARY
# =========================================================

con = duckdb.connect()

final_summary = con.execute(f"""
SELECT

    COUNT(*) AS user_day_rows,

    COUNT(
        DISTINCT user
    ) AS users,

    COUNT(
        DISTINCT day
    ) AS calendar_days,

    MIN(day) AS first_day,

    MAX(day) AS last_day,

    SUM(
        CASE
            WHEN email_event_count > 0
            THEN 1
            ELSE 0
        END
    ) AS email_active_user_days,

    SUM(
        CASE
            WHEN usb_copy_count > 0
            THEN 1
            ELSE 0
        END
    ) AS usb_copy_active_user_days,

    SUM(
        CASE
            WHEN www_upload_count > 0
            THEN 1
            ELSE 0
        END
    ) AS http_upload_active_user_days,

    SUM(
        CASE
            WHEN logon_count > 0
            THEN 1
            ELSE 0
        END
    ) AS logon_active_user_days

FROM read_parquet(
    '{user_day_path}'
)
""").df()

con.close()


print("=" * 100)
print("FINAL USER-DAY SUMMARY")
print("=" * 100)

print(
    final_summary.to_string(
        index=False
    )
)

print(
    "\n✅ SUCCESS — user_day_r62_v1 is complete and ready for the next stage."
)

FINAL USER-DAY SUMMARY
 user_day_rows  users  calendar_days  first_day   last_day  email_active_user_days  usb_copy_active_user_days  http_upload_active_user_days  logon_active_user_days
       1394010   4000            516 2010-01-02 2011-06-01               1381557.0                   111015.0                       60025.0               1393129.0

✅ SUCCESS — user_day_r62_v1 is complete and ready for the next stage.
